# Tech Challenge Fase 3 — Fine-tuning em GPU (Google Colab)

Caminho alternativo ao MLX, para reproduzir o treinamento em GPU NVIDIA.
Consome exatamente o mesmo dataset gerado pelo pipeline do repositorio.

**Runtime:** Ambiente de execucao > Alterar tipo de ambiente > GPU (T4 basta).


## 1. Clonar o repositorio e instalar as dependencias

In [ ]:
!git clone https://github.com/ca-ayumi/tech_challenge_fase3.git
%cd tech_challenge_fase3
!pip install -q -r requirements-treino-cuda.txt

## 2. Gerar os dados

O mesmo comando da maquina local: prontuarios sinteticos, banco SQLite e o
dataset de fine-tuning ja anonimizado e curado.

In [ ]:
!python -m assistente_medico.cli preparar --forcar

Conferindo o relatorio da curadoria:

In [ ]:
import json
relatorio = json.load(open("dados/processados/relatorio_dataset.json"))
print(json.dumps(relatorio["curadoria"], indent=2, ensure_ascii=False))
print("particoes:", relatorio["particoes"])

## 3. Inspecionar um exemplo do dataset

Vale conferir antes de treinar: o turno do usuario traz a pergunta mais os
contextos recuperados, e o turno do assistente traz a resposta na estrutura de
tres blocos.

In [ ]:
exemplo = json.loads(open("dados/processados/train.jsonl").readline())
for mensagem in exemplo["messages"]:
    print(f"--- {mensagem['role'].upper()} ---")
    print(mensagem["content"][:700])
    print()

## 4. Fine-tuning por LoRA

Backend `peft`: transformers + peft + trl. Os hiperparametros sao os mesmos do
caminho MLX, definidos em `assistente_medico/config.py`.

In [ ]:
!python -m assistente_medico.finetuning.treinar --backend peft --iteracoes 400

## 5. Avaliar

Compara o modelo ajustado com o modelo base sem adaptador, no conjunto de teste,
e roda o conjunto de red team de seguranca.

In [ ]:
!BACKEND_LLM=transformers python -m assistente_medico.finetuning.avaliar --comparar-base --backend transformers

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("avaliacao/resultados/avaliacao.md").read()))

## 6. Testar o assistente completo

O fluxo LangGraph inteiro: triagem, contexto do paciente, regras, recuperacao,
geracao, guardrails, explicabilidade e auditoria.

In [ ]:
import os
os.environ["BACKEND_LLM"] = "transformers"

from assistente_medico.grafo import AssistenteMedico, Dependencias
from assistente_medico.llm import criar_modelo_chat

assistente = AssistenteMedico(Dependencias(modelo=criar_modelo_chat("transformers")))

for pergunta in [
    "Quais os criterios para abrir o protocolo de sepse?",
    "Ha alguma pendencia no leito PS-07?",
    "Qual a dose de noradrenalina para esse paciente?",
]:
    estado = assistente.responder(pergunta)
    print("=" * 78)
    print("PERGUNTA:", pergunta)
    print("categoria:", estado["categoria"], "| etapas:", len(estado["etapas"]))
    print(estado["resposta"])

## 7. Baixar o adaptador treinado

O adaptador tem poucos megabytes e pode ser versionado junto do codigo.

In [ ]:
!zip -r adaptador-lora.zip modelos/adaptador-lora -x "*/checkpoints/*"
from google.colab import files
files.download("adaptador-lora.zip")